# Tushare stock data demo

Fetch daily stock prices, preview the data, and save a CSV. Run the cells from top to bottom.

In [ ]:
%pip install -q tushare pandas

### Connect

Get your token from [Tushare Pro](https://tushare.pro/document/1?doc_id=39). Set `TUSHARE_TOKEN` in your environment or enter it at the hidden prompt.

In [ ]:
import os
from getpass import getpass

import pandas as pd
import tushare as ts

token = os.getenv("TUSHARE_TOKEN") or getpass("Tushare token: ")
pro = ts.pro_api(token)

### Fetch and preview

Example: Ping An Bank (`000001.SZ`) in January 2025. Change the code or dates below; dates use `YYYYMMDD`.

The [`daily` endpoint](https://tushare.pro/document/2?doc_id=27) returns unadjusted prices. `vol` is in lots of 100 shares; `amount` is in thousands of CNY. Your account needs access to this endpoint.

In [ ]:
stock_code = "000001.SZ"
start_date = "20250101"
end_date = "20250131"

prices = pro.daily(
    ts_code=stock_code,
    start_date=start_date,
    end_date=end_date,
    fields="ts_code,trade_date,open,high,low,close,vol,amount",
)

if prices.empty:
    raise ValueError("No data returned. Check the stock code and date range.")

prices["trade_date"] = pd.to_datetime(prices["trade_date"], format="%Y%m%d")
prices = prices.sort_values("trade_date").reset_index(drop=True)

print(f"{len(prices)} trading days fetched.")
prices.head()

### Save a CSV

Save the full result in the current working directory.

In [ ]:
csv_path = f"{stock_code}_{start_date}_{end_date}.csv"
prices.to_csv(csv_path, index=False, date_format="%Y-%m-%d")
print(f"Saved to {csv_path}")